# Emissions Sensitivity Analysis

Breakdown of GHG emissions across sensitivity analyses.

- Row 1: Net emissions by gas (CO₂, CH₄, N₂O in CO₂eq terms) — stacked area
- Rows 2–4: Per-gas breakdown by emission source — stacked area

In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from sensitivity_utils import (
    FONTSIZE_TITLE,
    GAS_CMAPS,
    GAS_DISPLAY,
    PRETTY_NAMES_EMISSIONS,
    PRETTY_NAMES_GAS,
    extract_combined_scenarios,
    extract_scenarios_with_param,
    get_log_ticks,
    load_emissions_by_source,
    load_net_emissions_by_gas,
    plot_stacked_emissions,
    set_dual_xaxis_labels,
    set_dual_xlabel,
)

In [ ]:
# Configuration
PROJECT_ROOT = Path("..").resolve()
CONFIG_NAME = "sensitivity"
CACHE_DIR = Path("cache") / CONFIG_NAME

## Extract scenarios

In [ ]:
# YLL scenarios
yll_scenarios_all = extract_scenarios_with_param(
    PROJECT_ROOT,
    CONFIG_NAME,
    param_path=["health", "value_per_yll"],
    scenario_prefix="yll_",
)
yll_param_values_full = [p for p, _, _ in yll_scenarios_all]
yll_scenarios = [(p, s, f) for p, s, f in yll_scenarios_all if f.exists()]
print(f"YLL: {len(yll_scenarios)}/{len(yll_scenarios_all)} solved")

# GHG scenarios
ghg_scenarios_all = extract_scenarios_with_param(
    PROJECT_ROOT,
    CONFIG_NAME,
    param_path=["emissions", "ghg_price"],
    scenario_prefix="ghg_",
)
ghg_scenarios_all = [
    (p, s, f) for p, s, f in ghg_scenarios_all if not s.startswith("ghg_yll_")
]
ghg_param_values_full = [p for p, _, _ in ghg_scenarios_all]
ghg_scenarios = [(p, s, f) for p, s, f in ghg_scenarios_all if f.exists()]
print(f"GHG: {len(ghg_scenarios)}/{len(ghg_scenarios_all)} solved")

# Combined GHG+YLL scenarios
ghg_yll_scenarios_all = extract_combined_scenarios(
    PROJECT_ROOT,
    CONFIG_NAME,
    ghg_param_path=["emissions", "ghg_price"],
    yll_param_path=["health", "value_per_yll"],
    scenario_prefix="ghg_yll_",
)
ghg_yll_ghg_values_full = [ghg for ghg, _, _, _ in ghg_yll_scenarios_all]
ghg_yll_yll_values_full = [yll for _, yll, _, _ in ghg_yll_scenarios_all]
ghg_yll_scenarios_full = [
    (ghg, yll, s, f)
    for ghg, yll, s, f in ghg_yll_scenarios_all
    if f.exists() and "break" not in s
]
ghg_yll_scenarios = [(ghg, s, f) for ghg, yll, s, f in ghg_yll_scenarios_full]
ghg_yll_param_pairs = [(ghg, yll) for ghg, yll, s, f in ghg_yll_scenarios_full]
ghg_yll_ghg_values = [ghg for ghg, _ in ghg_yll_param_pairs]
ghg_yll_yll_values = [yll for _, yll in ghg_yll_param_pairs]
print(f"GHG+YLL: {len(ghg_yll_scenarios)}/{len(ghg_yll_scenarios_all)} solved")

## Load data

In [ ]:
# Net emissions by gas
df_yll_gas = load_net_emissions_by_gas(
    yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "yll_value"
)
df_ghg_gas = load_net_emissions_by_gas(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)
df_ghg_yll_gas = load_net_emissions_by_gas(
    ghg_yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)

print(f"YLL by-gas shape: {df_yll_gas.shape}")
print(df_yll_gas)
print()
print(f"GHG by-gas shape: {df_ghg_gas.shape}")
print(df_ghg_gas)
print()
print(f"Combined by-gas shape: {df_ghg_yll_gas.shape}")
print(df_ghg_yll_gas)

In [ ]:
# Per-source breakdown from network files
yll_by_source = load_emissions_by_source(
    yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "yll_value",
    cache_path=CACHE_DIR / "emissions_by_source_yll",
)
ghg_by_source = load_emissions_by_source(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
    cache_path=CACHE_DIR / "emissions_by_source_ghg",
)
ghg_yll_by_source = load_emissions_by_source(
    ghg_yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
    cache_path=CACHE_DIR / "emissions_by_source_ghg_yll",
)

for label, by_source in [
    ("YLL", yll_by_source),
    ("GHG", ghg_by_source),
    ("Combined", ghg_yll_by_source),
]:
    for gas in ["CO2", "CH4", "N2O"]:
        cols = list(by_source[gas].columns) if not by_source[gas].empty else []
        print(f"{label} {gas}: {cols}")

## Emissions sensitivity figure

In [ ]:
# X-axis configuration
YLL_XTICKS, YLL_XTICKLABELS = get_log_ticks(yll_param_values_full)
YLL_XLABEL = "Value per Year of Life Lost [USD/YLL]"

GHG_XTICKS, GHG_XTICKLABELS = get_log_ticks(ghg_param_values_full)
GHG_XLABEL = "GHG price [USD/tCO\u2082eq]"

GHG_YLL_XTICKS, GHG_YLL_XTICKLABELS = get_log_ticks(ghg_yll_ghg_values_full)
tick_to_yll = dict(zip(ghg_yll_ghg_values_full, ghg_yll_yll_values_full))
tick_to_yll.update(dict(zip(ghg_yll_ghg_values, ghg_yll_yll_values)))
GHG_YLL_GHG_VALUES = list(GHG_YLL_XTICKS)
GHG_YLL_YLL_VALUES = []
for tick, label in zip(GHG_YLL_XTICKS, GHG_YLL_XTICKLABELS):
    if label == "0":
        GHG_YLL_YLL_VALUES.append(0.0)
    elif tick in tick_to_yll:
        GHG_YLL_YLL_VALUES.append(tick_to_yll[tick])
    else:
        nonzero_pairs = [
            (g, y)
            for g, y in zip(ghg_yll_ghg_values_full, ghg_yll_yll_values_full)
            if g > 0
        ]
        ratio = nonzero_pairs[0][1] / nonzero_pairs[0][0] if nonzero_pairs else 1
        GHG_YLL_YLL_VALUES.append(tick * ratio)


# --- Colour assignments ---


def assign_gas_colors(df):
    """Assign colours to gas columns."""
    gas_colors = {
        GAS_DISPLAY["co2"]: "#636363",
        GAS_DISPLAY["ch4"]: "#31a354",
        GAS_DISPLAY["n2o"]: "#e6550d",
    }
    return {col: gas_colors.get(col, "#999999") for col in df.columns}


def assign_source_colors_for_gas(gas, all_sources):
    """Assign gas-specific colormap shades to source categories."""
    cmap = matplotlib.colormaps[GAS_CMAPS.get(gas, "Blues")]
    n = len(all_sources)
    colors = {}
    for i, src in enumerate(all_sources):
        shade = 0.3 + 0.55 * i / max(n - 1, 1)
        colors[src] = cmap(shade)
    return colors


def prepare_source_df(df, threshold=0.005):
    """Order columns by mean absolute value (largest first), drop near-zero."""
    if df.empty:
        return df
    significant = df.columns[df.abs().max() >= threshold]
    df = df[significant]
    # Separate positive-mean and negative-mean columns
    pos_cols = sorted(
        [c for c in df.columns if df[c].mean() >= 0],
        key=lambda c: df[c].mean(),
        reverse=True,
    )
    neg_cols = sorted(
        [c for c in df.columns if df[c].mean() < 0],
        key=lambda c: df[c].mean(),
    )
    return df[pos_cols + neg_cols]


# --- Prepare data ---
gas_colors = assign_gas_colors(df_yll_gas)

# Order gas columns: largest first
gas_order = df_yll_gas.mean().sort_values(ascending=False).index.tolist()
df_yll_gas_plot = df_yll_gas[gas_order]
df_ghg_gas_plot = df_ghg_gas[[c for c in gas_order if c in df_ghg_gas.columns]]
df_ghg_yll_gas_plot = df_ghg_yll_gas[
    [c for c in gas_order if c in df_ghg_yll_gas.columns]
]

# Prepare source DataFrames and build consistent colour maps per gas
source_dfs = {}
source_colors = {}
for gas in ["CO2", "CH4", "N2O"]:
    # Collect all source names across the three sensitivity runs
    all_sources_for_gas = set()
    for label, by_source in [
        ("yll", yll_by_source),
        ("ghg", ghg_by_source),
        ("ghg_yll", ghg_yll_by_source),
    ]:
        df_src = prepare_source_df(by_source[gas])
        source_dfs[(gas, label)] = df_src
        all_sources_for_gas.update(df_src.columns.tolist())
    source_colors[gas] = assign_source_colors_for_gas(gas, sorted(all_sources_for_gas))

# --- Build figure: 4 rows x 3 columns ---
fig, axes = plt.subplots(4, 3, figsize=(8, 9))

gases_for_rows = ["CO2", "CH4", "N2O"]
panel_labels = "abcdefghijkl"
panel_idx = 0

# --- Row 0: Net emissions by gas ---
for col, (df_gas, xlabel, x_ticks, x_ticklabels) in enumerate(
    [
        (df_yll_gas_plot, YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
        (df_ghg_gas_plot, GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
        (
            df_ghg_yll_gas_plot,
            "",
            GHG_YLL_XTICKS,
            [""] * len(GHG_YLL_XTICKS),
        ),
    ]
):
    if df_gas.empty:
        panel_idx += 1
        continue
    plot_stacked_emissions(
        df_gas,
        gas_colors,
        axes[0, col],
        xlabel=xlabel,
        ylabel="Net emissions [GtCO\u2082eq]" if col == 0 else "",
        panel_label=panel_labels[panel_idx],
        x_ticks=x_ticks,
        x_ticklabels=x_ticklabels,
        min_height_for_label=0.5,
        pretty_names=PRETTY_NAMES_GAS,
    )
    panel_idx += 1

# --- Rows 1-3: Per-gas source breakdown ---
for row_offset, gas in enumerate(gases_for_rows):
    row = row_offset + 1
    gas_display = GAS_DISPLAY[gas]
    for col, (label, xlabel, x_ticks, x_ticklabels) in enumerate(
        [
            ("yll", YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
            ("ghg", GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
            ("ghg_yll", "", GHG_YLL_XTICKS, [""] * len(GHG_YLL_XTICKS)),
        ]
    ):
        df_src = source_dfs[(gas, label)]
        if df_src.empty:
            panel_idx += 1
            continue
        colors_for_panel = {
            src: source_colors[gas][src]
            for src in df_src.columns
            if src in source_colors[gas]
        }
        plot_stacked_emissions(
            df_src,
            colors_for_panel,
            axes[row, col],
            xlabel=xlabel,
            ylabel=f"{gas_display} emissions [GtCO\u2082eq]" if col == 0 else "",
            panel_label=panel_labels[panel_idx],
            x_ticks=x_ticks,
            x_ticklabels=x_ticklabels,
            min_height_for_label=0.08,
            pretty_names=PRETTY_NAMES_EMISSIONS,
        )
        panel_idx += 1

# Column titles
axes[0, 0].set_title(
    "Health value sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 1].set_title(
    "GHG price sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 2].set_title(
    "Combined sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)

# Share y-axis within rows
for row in range(4):
    y_min = min(ax.get_ylim()[0] for ax in axes[row, :])
    y_max = max(ax.get_ylim()[1] for ax in axes[row, :])
    for col in range(3):
        axes[row, col].set_ylim(y_min, y_max)

# Remove x-labels and ticks except bottom row
for row in range(3):
    for col in range(3):
        axes[row, col].set_xlabel("")
        axes[row, col].set_xticklabels([])

# Dual x-axis labels on bottom-right
set_dual_xaxis_labels(
    axes[3, 2], GHG_YLL_XTICKS, GHG_YLL_GHG_VALUES, GHG_YLL_YLL_VALUES
)
set_dual_xlabel(axes[3, 2])

# Remove y-labels from cols 1 and 2
for row in range(4):
    for col in [1, 2]:
        axes[row, col].set_ylabel("")
        axes[row, col].set_yticklabels([])

fig.align_ylabels(axes[:, 0])
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)

# Save
output_dir = PROJECT_ROOT / "notebooks" / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "emissions_sensitivity.pdf"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Saved to: {output_path}")
plt.show()